In [128]:
import pandas as pd
import numpy as  np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [129]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

In [130]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)

In [131]:
df = pd.read_csv('../data/Telco_customer_churn.csv')
df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [132]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   str    
 1   Count              7043 non-null   int64  
 2   Country            7043 non-null   str    
 3   State              7043 non-null   str    
 4   City               7043 non-null   str    
 5   Zip Code           7043 non-null   int64  
 6   Lat Long           7043 non-null   str    
 7   Latitude           7043 non-null   float64
 8   Longitude          7043 non-null   float64
 9   Gender             7043 non-null   str    
 10  Senior Citizen     7043 non-null   str    
 11  Partner            7043 non-null   str    
 12  Dependents         7043 non-null   str    
 13  Tenure Months      7043 non-null   int64  
 14  Phone Service      7043 non-null   str    
 15  Multiple Lines     7043 non-null   str    
 16  Internet Service   7043 non-null   

In [133]:
df.isnull().sum()

CustomerID              0
Count                   0
Country                 0
State                   0
City                    0
Zip Code                0
Lat Long                0
Latitude                0
Longitude               0
Gender                  0
Senior Citizen          0
Partner                 0
Dependents              0
Tenure Months           0
Phone Service           0
Multiple Lines          0
Internet Service        0
Online Security         0
Online Backup           0
Device Protection       0
Tech Support            0
Streaming TV            0
Streaming Movies        0
Contract                0
Paperless Billing       0
Payment Method          0
Monthly Charges         0
Total Charges           0
Churn Label             0
Churn Value             0
Churn Score             0
CLTV                    0
Churn Reason         5174
dtype: int64

In [134]:
df.columns

Index(['CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code',
       'Lat Long', 'Latitude', 'Longitude', 'Gender', 'Senior Citizen',
       'Partner', 'Dependents', 'Tenure Months', 'Phone Service',
       'Multiple Lines', 'Internet Service', 'Online Security',
       'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV',
       'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method',
       'Monthly Charges', 'Total Charges', 'Churn Label', 'Churn Value',
       'Churn Score', 'CLTV', 'Churn Reason'],
      dtype='str')

In [135]:
features = [
    'Tenure Months',
    'Monthly Charges',
    'Total Charges'
]

X = df[features]
y = df['Churn Label']

print(X.isnull().sum())
print(y.isnull().sum())

Tenure Months      0
Monthly Charges    0
Total Charges      0
dtype: int64
0


In [136]:
data = df[features + ['Churn Label']].copy()


In [137]:
for col in features:
    data[col] = pd.to_numeric(
        data[col],
        errors='coerce'
    )
data.isnull().sum()

Tenure Months       0
Monthly Charges     0
Total Charges      11
Churn Label         0
dtype: int64

In [138]:
print("Before handling missing values ", data.shape)
data = data.dropna()
print("After handling the missing values", data.shape)

Before handling missing values  (7043, 4)
After handling the missing values (7032, 4)


In [139]:
X = data[features]
y = data['Churn Label']
# y.isnull().sum()
print(y.unique())

<ArrowStringArray>
['Yes', 'No']
Length: 2, dtype: str


In [140]:
y = y.map(
    {
        "No" : 0,
        "Yes" : 1
    }
)
# print("Target Distribution : ", y.value_counts())
print(y.isnull().sum())


0


In [141]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

In [142]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [143]:
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train_scaled, y_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.Doesn't affect :meth:`fit` method.",None
Name,Type,Value
"classes_ classes_: array of shape (n_classes,)Class labels known to the classifier","ndarray[int64](2,)","[0,1]"
"effective_metric_ effective_metric_: str or callbleThe distance metric used. It will be same as the `metric` parameteror a synonym of it, e.g. 'euclidean' if the `metric` parameter set to'minkowski' and `p` parameter set to 2.",str,'eu...an'


In [144]:
y_pred = model.predict(X_test_scaled)
y_pred

array([0, 0, 0, ..., 0, 0, 0], shape=(1407,))

In [145]:
accuracy = accuracy_score(y_test, y_pred)
cnm = confusion_matrix(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1Score = f1_score(y_test, y_pred)

print("Accuracy : ", accuracy)
print("Confusion Matrix : ", cnm)
print("Precision : ", precision)
print("Recall : ", recall)
print("F1 Score : ", f1Score)

Accuracy :  0.7619047619047619
Confusion Matrix :  [[891 142]
 [193 181]]
Precision :  0.5603715170278638
Recall :  0.4839572192513369
F1 Score :  0.5193687230989957


In [146]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.82      0.86      0.84      1033
           1       0.56      0.48      0.52       374

    accuracy                           0.76      1407
   macro avg       0.69      0.67      0.68      1407
weighted avg       0.75      0.76      0.76      1407



In [147]:
y_prob = model.predict_proba(X_test_scaled)
print(y_prob[:10])
churn_probability = y_prob[:, 1]
print(churn_probability[:10])


[[1.  0. ]
 [0.6 0.4]
 [1.  0. ]
 [1.  0. ]
 [0.8 0.2]
 [0.8 0.2]
 [0.8 0.2]
 [1.  0. ]
 [0.4 0.6]
 [1.  0. ]]
[0.  0.4 0.  0.  0.2 0.2 0.2 0.  0.6 0. ]


In [148]:
k_values = [1, 3, 5, 7, 9, 11, 15, 21]
print("PERFORMANCE FOR DIFFERENT K VALUES")

print("==========================================")


for k in k_values:

    model = KNeighborsClassifier(
        n_neighbors=k
    )


    model.fit(
        X_train_scaled,
        y_train
    )


    predictions = model.predict(
        X_test_scaled
    )


    score = f1_score(
        y_test,
        predictions
    )


    print(
        "K =", k,
        "| F1 =", score
    )


PERFORMANCE FOR DIFFERENT K VALUES
K = 1 | F1 = 0.480946123521682
K = 3 | F1 = 0.49157303370786515
K = 5 | F1 = 0.5193687230989957
K = 7 | F1 = 0.5161290322580645
K = 9 | F1 = 0.5225225225225225
K = 11 | F1 = 0.5157421289355323
K = 15 | F1 = 0.5151515151515151
K = 21 | F1 = 0.521870286576169


In [150]:
paramGrid = {
    "n_neighbors" : [
        1,
        3,
        5,
        7,
        9,
        11,
        15,
        21
    ]
}

grid_model = GridSearchCV(
    estimator=KNeighborsClassifier(),
    param_grid=paramGrid,
    cv = 5,
    scoring="f1"
)

grid_model.fit(X_train_scaled, y_train)
grid_model.best_params_

{'n_neighbors': 7}

In [154]:
best_k = grid_model.best_params_["n_neighbors"]
print(best_k)
print("Best CV", grid_model.best_estimator_)

7
Best CV KNeighborsClassifier(n_neighbors=7)


In [155]:
best_estimator = grid_model.best_estimator_
y_pred_best = grid_model.predict(X_test_scaled)

In [156]:
print("Accuracy : ", accuracy_score(y_test, y_pred_best))
print("Precision : ", precision_score(y_test, y_pred_best))
print("Recall : ", recall_score(y_test, y_pred_best))
print("F1 Score : ", f1_score(y_test, y_pred_best))

Accuracy :  0.7654584221748401
Precision :  0.5714285714285714
Recall :  0.47058823529411764
F1 Score :  0.5161290322580645


In [157]:
print(classification_report(y_test, y_pred_best))

              precision    recall  f1-score   support

           0       0.82      0.87      0.85      1033
           1       0.57      0.47      0.52       374

    accuracy                           0.77      1407
   macro avg       0.70      0.67      0.68      1407
weighted avg       0.75      0.77      0.76      1407



In [158]:
final_prob = best_estimator.predict_proba(X_test_scaled)
print(final_prob[:10])

[[1.         0.        ]
 [0.57142857 0.42857143]
 [1.         0.        ]
 [0.85714286 0.14285714]
 [0.85714286 0.14285714]
 [0.57142857 0.42857143]
 [0.85714286 0.14285714]
 [1.         0.        ]
 [0.42857143 0.57142857]
 [1.         0.        ]]


In [159]:
final_churn_probability = (
    final_prob[:, 1]
)


print("\nFirst 10 Final Churn Probabilities:")

print(
    final_churn_probability[:10]
)


First 10 Final Churn Probabilities:
[0.         0.42857143 0.         0.14285714 0.14285714 0.42857143
 0.14285714 0.         0.57142857 0.        ]
